Prueba

In [1]:
pip install hyperopt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\Desktop\\pruebas_collab\\datosNarmax\\1pasos_lstm_pollution.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd,e
date,,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048,NaN
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575,NaN
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103,NaN
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962,NaN
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489,NaN


In [5]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [6]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas LSTM
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades LSTM
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes LSTM, es decir [observaciones, retardos, caracteristicas]

In [7]:
futuros = 1
pasados  = 12

In [8]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 0])


In [9]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43788, 12, 7)
Dimensiones de Y: (43788, 1)


In [10]:
print(datosX[0])

[[ 0.31768099 -1.2140229  -1.26852411  0.32968671 -0.38094383 -0.46404777
          nan]
 [ 0.52615226 -1.14430217 -1.26852411  0.32968671 -0.38094383 -0.44657536
          nan]
 [ 0.64684616 -0.86541928 -1.34931411  0.42612698 -0.38094383 -0.42910295
          nan]
 [ 0.88823396 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.39396181
          nan]
 [ 0.41643054 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.3764894
          nan]
 [ 0.09823754 -0.58653639 -1.4301041   0.52256725 -0.38094383 -0.35901698
          nan]
 [ 0.05434885 -0.58653639 -1.4301041   0.61900753 -0.38094383 -0.32387584
          nan]
 [ 0.26282012 -0.58653639 -1.34931411  0.7154478  -0.38094383 -0.2887347
          nan]
 [ 0.21893143 -0.65625711 -1.4301041   0.7154478  -0.38094383 -0.25359356
          nan]
 [ 0.3505975  -0.58653639 -1.34931411  0.81188808 -0.38094383 -0.21845241
          nan]
 [ 0.43837488 -0.58653639 -1.34931411  0.90832835 -0.38094383 -0.15700449
          nan]
 [ 0.57004095 -0.656257

Se dividen nuevamente los conjuntos de datos

In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30651, 12, 7)
Las dimensiones de testX son:  (8801, 12, 7)
Las dimensiones de valX son:  (4336, 12, 7)


In [12]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30651, 1)
Las dimensiones de testY son:  (8801, 1)
Las dimensiones de valY son:  (4336, 1)


Se crean métricas para medir desempeño

In [13]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [14]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [15]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1], testX.shape[2])))
    if (params['layers'] == 1):
      model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(LSTM(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=params['epochs'],
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [16]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

193/193 - 12s - 65ms/step - ia: 0.3402 - loss: 0.8512 - mae: 0.6878 - rmse: 0.8956 - smape: 1.4568 - val_ia: 0.2865 - val_loss: 0.3500 - val_mae: 0.4493 - val_rmse: 0.5368 - val_smape: 1.1399

Epoch 2/128                                           

193/193 - 3s - 17ms/step - ia: 0.5430 - loss: 0.6167 - mae: 0.5640 - rmse: 0.7655 - smape: 1.0763 - val_ia: 0.2901 - val_loss: 0.3389 - val_mae: 0.4370 - val_rmse: 0.5241 - val_smape: 1.0916

Epoch 3/128                                           

193/193 - 3s - 16ms/step - ia: 0.5741 - loss: 0.5602 - mae: 0.5394 - rmse: 0.7310 - smape: 1.0231 - val_ia: 0.2909 - val_loss: 0.3288 - val_mae: 0.4310 - val_rmse: 0.5167 - val_smape: 1.0759

Epoch 4/128                                           

193/193 - 3s - 18ms/step - ia: 0.5925 - loss: 0.5369 - mae: 0.5245 - rmse: 0.7135 - smape: 0.9972 - val_ia: 0.2914 - val_loss: 0.3173 - val_mae: 0.4266 - val_rmse: 0.5103 - val_smape: 1.0762

Epoch 5

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                           

25/25 - 15s - 591ms/step - ia: 0.1760 - loss: 1.0276 - mae: 0.7542 - rmse: 0.9869 - smape: 1.7028 - val_ia: 0.2986 - val_loss: 0.3974 - val_mae: 0.4804 - val_rmse: 0.6249 - val_smape: 1.2368

Epoch 2/16                                                                           

25/25 - 4s - 176ms/step - ia: 0.5982 - loss: 0.5481 - mae: 0.5254 - rmse: 0.7232 - smape: 0.9614 - val_ia: 0.5192 - val_loss: 0.2878 - val_mae: 0.3847 - val_rmse: 0.5289 - val_smape: 0.9566

Epoch 3/16                                                                           

25/25 - 4s - 165ms/step - ia: 0.6885 - loss: 0.3960 - mae: 0.4419 - rmse: 0.6208 - smape: 0.8419 - val_ia: 0.5458 - val_loss: 0.2541 - val_mae: 0.3736 - val_rmse: 0.4942 - val_smape: 0.9812

Epoch 4/16                                                                           

25/25 - 4s - 176ms/step - ia: 0.7455 - loss: 0.2902 - mae: 0.3748 - rmse: 0.53

...

49/49 - 6s - 124ms/step - ia: 0.2303 - loss: 0.9885 - mae: 0.7462 - rmse: 0.9861 - smape: 1.5171 - val_ia: 0.3452 - val_loss: 0.4276 - val_mae: 0.5249 - val_rmse: 0.6269 - val_smape: 1.4786

Epoch 2/16                                                                           

49/49 - 0s - 10ms/step - ia: 0.3481 - loss: 0.7912 - mae: 0.6645 - rmse: 0.8839 - smape: 1.3317 - val_ia: 0.4034 - val_loss: 0.3254 - val_mae: 0.4542 - val_rmse: 0.5495 - val_smape: 1.2259

Epoch 3/16                                                                           

49/49 - 1s - 11ms/step - ia: 0.5272 - loss: 0.5553 - mae: 0.5464 - rmse: 0.7338 - smape: 1.0506 - val_ia: 0.4773 - val_loss: 0.2443 - val_mae: 0.3788 - val_rmse: 0.4763 - val_smape: 0.9767

Epoch 4/16                                                                           

49/49 - 0s - 10ms/step - ia: 0.6543 - loss: 0.4131 - mae: 0.4544 - rmse: 0.6367 - smape: 0.8514 - val_ia: 0.5259 - val_loss: 0.2169 - val_mae: 0.3408 - val_rmse: 0.4428

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



385/385 - 8s - 22ms/step - ia: 0.4323 - loss: 0.6981 - mae: 0.6164 - rmse: 0.7942 - smape: 1.2562 - val_ia: 0.2788 - val_loss: 0.2931 - val_mae: 0.4073 - val_rmse: 0.4690 - val_smape: 1.0956

Epoch 2/256                                                                          

385/385 - 3s - 8ms/step - ia: 0.6328 - loss: 0.4137 - mae: 0.4657 - rmse: 0.6166 - smape: 0.9276 - val_ia: 0.2983 - val_loss: 0.2498 - val_mae: 0.3702 - val_rmse: 0.4342 - val_smape: 0.9992

Epoch 3/256                                                                          

385/385 - 3s - 8ms/step - ia: 0.6738 - loss: 0.3631 - mae: 0.4256 - rmse: 0.5750 - smape: 0.8474 - val_ia: 0.3134 - val_loss: 0.2244 - val_mae: 0.3468 - val_rmse: 0.4107 - val_smape: 0.9370

Epoch 4/256                                                                          

385/385 - 3s - 7ms/step - ia: 0.7006 - loss: 0.3102 - mae: 0.3932 - rmse: 0.5305 - smape: 0.7969 - val_ia: 0.3277 - val_loss: 0.2013 - val_mae: 0.3234 - val_rmse: 0.

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



770/770 - 11s - 14ms/step - ia: 0.7396 - loss: 0.2428 - mae: 0.3216 - rmse: 0.4171 - smape: 0.6596 - val_ia: 0.4315 - val_loss: 0.0754 - val_mae: 0.1791 - val_rmse: 0.2166 - val_smape: 0.5418

Epoch 2/32                                                                           

770/770 - 5s - 7ms/step - ia: 0.8159 - loss: 0.1204 - mae: 0.2342 - rmse: 0.3117 - smape: 0.5012 - val_ia: 0.4561 - val_loss: 0.0671 - val_mae: 0.1647 - val_rmse: 0.2051 - val_smape: 0.4949

Epoch 3/32                                                                           

770/770 - 5s - 7ms/step - ia: 0.8274 - loss: 0.1077 - mae: 0.2177 - rmse: 0.2931 - smape: 0.4730 - val_ia: 0.4939 - val_loss: 0.0582 - val_mae: 0.1419 - val_rmse: 0.1822 - val_smape: 0.4501

Epoch 4/32                                                                           

770/770 - 6s - 7ms/step - ia: 0.8359 - loss: 0.1006 - mae: 0.2100 - rmse: 0.2810 - smape: 0.4561 - val_ia: 0.4908 - val_loss: 0.0569 - val_mae: 0.1407 - val_rmse: 0

In [17]:
print(best)

{'activation': 1, 'batch': 1, 'dropout': 0.0, 'epochs': 5, 'layers': 2.0, 'learning_rate': 0.000637222381143688, 'units': 1}


In [18]:
print(best)

{'activation': 1, 'batch': 1, 'dropout': 0.0, 'epochs': 5, 'layers': 2.0, 'learning_rate': 0.000637222381143688, 'units': 1}
